# CRM сегменты: визуализации для бизнеса

Отдельная тетрадка с двумя правками:
- динамика ФП/СФП по месяцам с выделением границ годов полупрозрачными пунктирными линиями;
- круговая диаграмма уникальных ИНН по сегментам.

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def add_year_separators(ax, dt_index):
    years = sorted(dt_index.year.unique())
    for year in years[1:]:
        boundary = pd.Timestamp(f"{year}-01-01")
        ax.axvline(boundary, color="gray", linestyle="--", linewidth=1.2, alpha=0.4)


In [ ]:
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
if "VAL" in df.columns:
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()

df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
df = df[df["segment"].notna()].copy()

df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
df["year_month"] = df["dt"].dt.to_period("M")

print(f"Готово к анализу: {len(df):,} строк")
print(f"Уникальных ИНН: {df['inn'].nunique():,}")

## Уникальные ИНН по сегментам: круговая диаграмма

In [ ]:
total_inn = df["inn"].nunique()
inn_by_seg = df.groupby("segment")["inn"].nunique().reindex(SEG_ORDER, fill_value=0)
inn_share = (inn_by_seg / total_inn * 100).round(1)

tbl_inn = pd.DataFrame({
    "Сегмент": SEG_ORDER,
    "Уникальных ИНН": [inn_by_seg[s] for s in SEG_ORDER],
    "Доля, %": [inn_share[s] for s in SEG_ORDER],
})
display(tbl_inn.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(7, 7))
labels = [f"{seg} ({inn_by_seg[seg]:,})" for seg in SEG_ORDER]
ax.pie(
    inn_by_seg.values,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
    textprops={"fontsize": 10}
)
ax.set_title("Уникальные ИНН по сегментам", fontsize=13, fontweight="bold")
ax.axis("equal")
plt.tight_layout()
plt.show()

## Динамика ФП/СФП с выделением границ годов

In [ ]:
def plot_monthly_dynamics(data, fp_type, title):
    subset = data[data["TYPE_FP"] == fp_type].copy()
    pivot = subset.groupby(["year_month", "segment"]).size().unstack(fill_value=0)
    pivot = pivot.reindex(columns=SEG_ORDER, fill_value=0)
    pivot.index = pivot.index.to_timestamp()

    fig, ax = plt.subplots(figsize=(14, 5))
    pivot.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)
    add_year_separators(ax, pivot.index)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Месяц")
    ax.set_ylabel(f"Количество {fp_type}")
    ax.legend(title="Сегмент")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


plot_monthly_dynamics(df, "ФП", "Количество выявленных ФП по месяцам (2023-2025)")
plot_monthly_dynamics(df, "СФП", "Количество выявленных СФП по месяцам (2023-2025)")